# 🩺 Protocolos Clínicos e Diretrizes Terapêuticas - PCDT

O script abaixo extrai todos os Protocolos Clínicos e Diretrizes Terapêuticas (PCDT) contidos no portal do Ministério da Saúde (https://www.gov.br/saude/pt-br/assuntos/pcdt).

Os PCDTs são diretrizes clínicas do Sistema Único de Saúde (SUS) que norteiam condutas no cuidado às doenças e agravos, auxiliando médicos, profissionais de saúde, pacientes, estabelecimentos e serviços de saúde referenciais e os gestores em saúde.

## 📥 Crawler e Downloader

In [4]:
import os
import time
import requests
from bs4 import BeautifulSoup
import string
from urllib.parse import urljoin, unquote
import urllib3
import re

# Desabilita avisos de certificado SSL (comum no Gov.br)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.gov.br/saude/pt-br/assuntos/pcdt/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
MAIN_DIR = "PCDTs_Ministerio_da_Saude"

def limpar_nome_arquivo(nome):
    """Remove caracteres que não são permitidos em nomes de arquivos pelo Windows/Linux."""
    return re.sub(r'[\\/*?:"<>|]', "", nome)

def extrair_nome_arquivo(response, nome_padrao):
    """Extrai e decodifica o nome do arquivo enviado pelo cabeçalho do Plone."""
    cd = response.headers.get("Content-Disposition", "")
    if not cd:
        return nome_padrao + ".pdf"

    # Tenta capturar no padrão codificado do Plone: filename*=UTF-8''nome%20com%20espaco.pdf
    match_utf8 = re.search(r"filename\*=UTF-8''(.+)", cd, re.IGNORECASE)
    if match_utf8:
        nome_decodificado = unquote(match_utf8.group(1)) # Transforma %20 em espaço, etc.
        return limpar_nome_arquivo(nome_decodificado)

    # Tenta capturar no padrão normal: filename="nome_do_arquivo.pdf"
    match_normal = re.search(r'filename=["\']?([^"\';]+)["\']?', cd, re.IGNORECASE)
    if match_normal:
        return limpar_nome_arquivo(match_normal.group(1).strip())

    return nome_padrao + ".pdf"

def main():
    if not os.path.exists(MAIN_DIR):
        os.makedirs(MAIN_DIR)

    letras = string.ascii_lowercase

    for letra in letras:
        url_letra = urljoin(BASE_URL, f"{letra}/")
        url_letra_limpa = url_letra.rstrip("/")

        letra_dir = os.path.join(MAIN_DIR, letra)
        if not os.path.exists(letra_dir):
            os.makedirs(letra_dir)

        print(f"\n[{letra.upper()}] Acessando página: {url_letra}")

        try:
            response = requests.get(url_letra, headers=HEADERS, timeout=15, verify=False)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Erro ao acessar a página '{letra}': {e}")
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        links_documentos = set()

        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"]

            # Limpa parâmetros de busca e âncoras da URL
            clean_href = href.split("?")[0].split("#")[0].rstrip("/")

            # Se o Plone jogou um /view no final do link, nós o removemos
            if clean_href.endswith("/view"):
                clean_href = clean_href[:-5]

            # Se o link já veio com /@@download/file, nós o removemos temporariamente para padronizar
            if clean_href.endswith("/@@download/file"):
                clean_href = clean_href.replace("/@@download/file", "")

            # Garante que o link é um "filho" da letra (ex: ../pcdt/a/doenca) e não a própria letra (../pcdt/a)
            if clean_href.startswith(url_letra_limpa + "/") and len(clean_href) > len(url_letra_limpa) + 1:
                links_documentos.add(clean_href)

        if not links_documentos:
            print(f"Nenhum documento encontrado na letra '{letra}'.")
            continue

        # Inicia o download
        for base_link in sorted(links_documentos):
            download_url = f"{base_link}/@@download/file"
            slug_doenca = base_link.split('/')[-1]

            print(f"  - Obtendo: {slug_doenca} ...", end=" ", flush=True)

            try:
                pdf_response = requests.get(download_url, headers=HEADERS, timeout=30, stream=True, verify=False)

                if pdf_response.status_code == 200:
                    content_type = pdf_response.headers.get('Content-Type', '').lower()

                    if 'text/html' not in content_type:
                        nome_arquivo = extrair_nome_arquivo(pdf_response, slug_doenca)

                        if not nome_arquivo.lower().endswith('.pdf'):
                            nome_arquivo += '.pdf'

                        caminho_arquivo = os.path.join(letra_dir, nome_arquivo)

                        if os.path.exists(caminho_arquivo):
                            print(f"Já existe! (Pulando)")
                            continue

                        with open(caminho_arquivo, "wb") as f:
                            for chunk in pdf_response.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        print(f"OK! (Salvo: {nome_arquivo})")
                    else:
                        print("FALHA! (Retornou HTML - Possível subpasta sem arquivo direto)")
                else:
                    print(f"FALHA! (HTTP {pdf_response.status_code})")

            except requests.RequestException as e:
                print(f"ERRO! ({e})")

            time.sleep(0.5)

if __name__ == "__main__":
    main()


[A] Acessando página: https://www.gov.br/saude/pt-br/assuntos/pcdt/a/
  - Obtendo: a ... FALHA! (HTTP 404)
  - Obtendo: acidente-vascular-cerebral-isquemico-agudo ... OK! (Salvo: Acidente Vascular Cerebral Isquêmico Agudo - Portaria Conjunta SAES-SECTICS n 29.pdf)
  - Obtendo: acidentes-escorpionicos ... OK! (Salvo: PCDT - Acidentes Escorpiônicos.pdf)
  - Obtendo: acidentes-ofidicos ... FALHA! (HTTP 404)
  - Obtendo: acromegalia.pdf ... OK! (Salvo: Acromegalia - PCDT.pdf)
  - Obtendo: adenocarcinoma-de-colon-e-de-reto ... OK! (Salvo: Adenocarcinoma de Cólon e de Reto - PCDT.pdf)
  - Obtendo: adenocarcinoma-de-estomago.pdf ... OK! (Salvo: Adenocarcinoma de Estomago.pdf)
  - Obtendo: adenocarcinoma-prostata.pdf ... OK! (Salvo: pcdt-Adenocarcinoma-Prostata.pdf)
  - Obtendo: amiloidoses-associadas-a-transtirretina.pdf ... OK! (Salvo: Amiloidoses Associadas a Transtirretina.pdf)
  - Obtendo: anemia-deficiencia-de-ferro ... OK! (Salvo: Anemia Deficiencia de Ferro.pdf)
  - Obtendo: anemia-he

## 📦 Compactação e Download

In [5]:
import shutil
from google.colab import files
import os

MAIN_DIR = "PCDTs_Ministerio_da_Saude"

if os.path.exists(MAIN_DIR):
    print("Compactando os arquivos... Isso pode levar alguns instantes.")
    # Compacta a pasta inteira
    shutil.make_archive('PCDTs_Completos', 'zip', MAIN_DIR)

    print("Iniciando o download do arquivo ZIP...")
    # Solicita ao Colab que empurre o arquivo para o seu PC
    files.download('PCDTs_Completos.zip')
else:
    print("A pasta de PDFs não foi encontrada. Certifique-se de que o Bloco 1 rodou com sucesso!")

Compactando os arquivos... Isso pode levar alguns instantes.
Iniciando o download do arquivo ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>